In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV


In [12]:
df= pd.read_excel('/transit_simulator_edited.xlsx')

df.head()

,Date,BART,MUNI,Ferry,Bridge,Visits,Month
0,2020-01-01,2083734,8884200,420907,5420879,4489375,1
1,2020-02-01,1953373,8429898,408246,5222944,4210672,2
2,2020-03-01,804768,6322593,158585,3848968,2060361,3
3,2020-04-01,70475,1342800,5591,2506819,565890,4
4,2020-05-01,84218,1657400,7067,3443831,772125,5


In [13]:
DF2= df[['BART', 'MUNI', 'Ferry', 'Bridge', 'Visits']]

In [14]:
df['ln_visits']= np.log(df['Visits'])
df['ln_bart']= np.log(df['BART'])
df['ln_muni']= np.log(df['MUNI'])
df['ln_ferry']= np.log(df['Ferry'])
df['ln_bridge']= np.log(df['Bridge'])

# LASSO Regression

In [26]:
month_dummies= pd.get_dummies(df['Month'], drop_first=True).astype(int)
X= pd.concat([month_dummies, df[['ln_bart', 'ln_muni', 'ln_ferry', 'ln_bridge']]], axis=1)

X = sm.add_constant(X)
y= df['ln_visits']

In [27]:
enet_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNetCV(
        l1_ratio=[0.6, 0.4],
        alphas=np.logspace(-3, 1, 50),
        cv=5,
        max_iter=100000,
        random_state=42
    ))
])

X.columns = X.columns.astype(str)

enet_pipe.fit(X, y)
enet = enet_pipe.named_steps["enet"]
enet_df = pd.DataFrame({"variable": X.columns, "coef": enet.coef_}) \
             .sort_values("coef", key=abs, ascending=False)
print(f"Chosen l1_ratio={enet.l1_ratio_}, alpha={enet.alpha_}")
print(enet_df[enet_df.coef != 0])

Chosen l1_ratio=0.6, alpha=0.0012067926406393288
     variable      coef
12    ln_bart  0.324746
15  ln_bridge  0.075517
8           9 -0.012902
14   ln_ferry  0.011043
6           7  0.010161
10         11 -0.007801
9          10 -0.007428
5           6  0.005858
7           8 -0.005565
1           2  0.005433
11         12  0.005390
13    ln_muni  0.002489
3           4 -0.001781
2           3 -0.001295
4           5 -0.001166
